[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day4_lecture.ipynb)

# Day 4 · 강의 — 딥러닝

손글씨 숫자로 배우고 동물 사진으로 확인한다

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

수업을 따라가며 진행한다.

**실습** 셀은 그대로 실행해 결과를 눈으로 확인한다.
**문제** 셀은 수업 중에 같이 푼다.

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 이미지를 텐서로

### 준비

아래 셀을 **먼저 한 번** 실행한다. 자료를 받고 학습·평가 함수를 만든다.
내려받기와 기본 모델 학습까지 1분쯤 걸린다.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

train = datasets.MNIST('data', train=True,  download=True, transform=transforms.ToTensor())
test  = datasets.MNIST('data', train=False, download=True, transform=transforms.ToTensor())
loader      = DataLoader(train, batch_size=128, shuffle=True)
test_loader = DataLoader(test,  batch_size=1000)

def fit(model, ld=None, epochs=3, lr=0.001):
    """학습 루프 다섯 줄을 함수로 묶어 둔 것"""
    ld = ld if ld is not None else loader
    torch.manual_seed(42)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for xb, yb in ld:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
    return model

def score(model, ld=None):
    """학습에 안 쓴 자료로 재는 정확도"""
    ld = ld if ld is not None else test_loader
    correct = total = 0
    with torch.no_grad():
        for xb, yb in ld:
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += len(yb)
    return correct / total

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128), nn.ReLU(),
    nn.Linear(128, 64),  nn.ReLU(),
    nn.Linear(64, 10))
model = fit(model, epochs=3)
xb, yb = next(iter(loader))
print('준비 끝 — 기본 모델 정확도', round(score(model), 4))

**실습.** 받아 온 자료를 들여다본다.

In [ ]:
x, y = train[0]
print(len(train), len(test))
print(x.shape, x.dtype, y)
print(x.min().item(), x.max().item())

> **실습문제 1.** 그림 한 장을 **글자로 찍어 본다.** 밝기가 0.5 를 넘는 칸은 `#`, 나머지는 공백으로 28줄을 출력한다.

In [ ]:
x, y = train[0]
a = x[0]
# 여기에 작성한다


print('정답은', y)

> **실습문제 2.** `x` 를 **한 줄로 펴서** `flat` 에 담고 모양을 확인한다.

In [ ]:
x, y = train[0]
# 여기에 작성한다


assert tuple(flat.shape) == (784,), f'기대 (784,), 실제 {tuple(flat.shape)}'
print('통과 —', flat.shape)

## 2. 모델 만들기

**실습.** 층을 순서대로 적기만 하면 모델이 된다. 준비 셀에서 만든 model 이 이 모양이다.

In [ ]:
print(model)
print('계수', sum(p.numel() for p in model.parameters()))

**실습.** 흐름에 손을 대야 할 때는 nn.Module 을 상속해 forward 에 직접 쓴다.

In [ ]:
class Net(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.fc1 = nn.Linear(784, hidden)
        self.fc2 = nn.Linear(hidden, 10)

    def forward(self, x):
        x = x.flatten(1)
        return self.fc2(torch.relu(self.fc1(x)))

net = Net()
print(net(xb).shape)

> **실습문제 3.** **층 하나짜리** 모델을 만들어 `flat_model` 에 담는다. 펴고 나서 곧바로 열 갈래로 보낸다.

In [ ]:
# 여기에 작성한다


assert flat_model(xb).shape == (128, 10), f'실제 {tuple(flat_model(xb).shape)}'
print('계수', sum(p.numel() for p in flat_model.parameters()))

## 3. 학습

**실습.** 다섯 줄이 한 걸음이다. 배치가 469개니 1 에폭에 469 걸음이다.

In [ ]:
print('배치', len(loader), '개 · 3 에폭이면', len(loader) * 3, '걸음')
print('준비 셀에서 이미 3 에폭 돌렸다 — 정확도', round(score(model), 4))

> **실습문제 4.** **학습 루프 다섯 줄을 순서대로** 쓴다. `flat_model` 을 1 에폭 돌린 뒤 마지막 손실을 찍는다.
> 비우기 → 예측 → 손실 → 역전파 → 갱신.

In [ ]:
opt = torch.optim.Adam(flat_model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for xb2, yb2 in loader:
# 여기에 작성한다


acc = score(flat_model)
assert acc > 0.85, f'0.85 는 넘어야 한다: {acc}'
print('통과 — 층 하나짜리 정확도', round(acc, 4))

## 4. 평가와 진단

**실습.** 정확도는 학습에 안 쓴 1만 장으로 잰다. no_grad 로 감싼다.

In [ ]:
print('은닉 둘 + ReLU', round(score(model), 4))
print('층 하나       ', round(score(flat_model), 4))

> **실습문제 5.** **활성화 함수가 없으면** 층을 쌓아도 소용없다는 것을 확인한다. `no_relu` 를 3 에폭 돌려 정확도를 `acc_no_relu` 에 담는다.

In [ ]:
# 여기에 작성한다


print('활성화 없음', round(acc_no_relu, 4))
print('층 하나  ', round(score(flat_model), 4))
print('둘이 비슷하면 쌓기만 해서는 얻는 것이 없다는 뜻이다')

## 5. 동물 사진으로 갈아 끼우기

**실습.** CIFAR-10 은 32×32 컬러 사진 6만 장이다. 열 갈래 중 여섯이 동물이다.

In [ ]:
c_train = datasets.CIFAR10('data', train=True,  download=True, transform=transforms.ToTensor())
c_test  = datasets.CIFAR10('data', train=False, download=True, transform=transforms.ToTensor())
c_loader      = DataLoader(c_train, batch_size=128, shuffle=True)
c_test_loader = DataLoader(c_test,  batch_size=1000)

print(c_train.classes)
print('한 장', c_train[0][0].shape, '→ 펴면 3 × 32 × 32 =', 3 * 32 * 32, '칸')

> **실습문제 6.** MNIST 에 쓰던 모델을 **입력 칸 수만 바꿔** 동물용으로 만든다. `animal` 에 담는다.
> 3 × 32 × 32 가 몇 칸인지 먼저 세어 본다.

In [ ]:
# 여기에 작성한다


cb, _ = next(iter(c_loader))
assert animal(cb).shape == (128, 10), f'실제 {tuple(animal(cb).shape)}'
print('통과 — 계수', sum(p.numel() for p in animal.parameters()))